In [2]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [5]:
@tool
def respostas_simples(pergunta: str) -> str:
    """Uma ferramenta simples para responder perguntas diretas."""
    return f"A resposta para '{pergunta}' é Paris."

tools = [respostas_simples]

In [6]:
agent = create_agent(tools=tools, model=llm, system_prompt="Você é um assistente prestativo. Use as ferramentas quando necessário.")

input_query = "Qual é a capital da França?"
response = agent.invoke(
    {"messages": [{"role": "user", "content": input_query}]}
)

print(response["messages"][-1].content)

A capital da França é **Paris**.


In [7]:
import sqlite3
from langchain.agents import create_agent
from langchain_core.tools import tool

In [15]:
def create_database():
    conn = sqlite3.connect(":memory:")
    c = conn.cursor()
    c.execute("""CREATE TABLE users (id INTEGER PRIMARY KEY, name TEXT, age INTEGER)""")
    c.execute("""INSERT INTO users (name, age) VALUES ('João', 25), ('Maria', 30)""")
    conn.commit()
    return conn

@tool
def search_user_db(user_name: str):
    """Procura informações sobre um usuário na base de dados"""
    conn = create_database()
    c = conn.cursor()
    c.execute("SELECT name, age FROM users WHERE name = ?", (user_name, ))
    result = c.fetchone()
    if result:
        return f"{result[0]} tem {result[1]} anos"
    else:
        return "Usuário não encontrado"

@tool
def sum_numbers(numbers):
    """Calcula a soma de uma lista de números"""
    if isinstance(numbers, str):
        numbers = eval(numbers)

    if isinstance(numbers, list):
        total = sum(numbers)
        return f"A soma dos números {numbers} é {total}"
    else:
        return "Erro: A entrada não é uma lista de números válida"

In [16]:
tools = [search_user_db, sum_numbers]

In [18]:
agent = create_agent(tools=tools, model=llm, system_prompt="Você é um assitente prestativo. Use as ferramentas quando necessário")

input_query = "Qual é a soma de 5, 10 e 15? Quantos anos Maria tem?"

response = agent.invoke(
    {"messages": [{"role": "user", "content": input_query}]}
)

print(response["messages"][-1].content)
for msg in response["messages"]:
    msg.pretty_print()

A soma de 5, 10 e 15 é **30**.

A Maria tem **30 anos**.
================================ Human Message =================================

Qual é a soma de 5, 10 e 15? Quantos anos Maria tem?
================================== Ai Message ==================================
Tool Calls:
  search_user_db (fc_a29bedca-7b8a-4cab-88d4-bd37c7b61167)
 Call ID: fc_a29bedca-7b8a-4cab-88d4-bd37c7b61167
  Args:
    user_name: Maria
================================= Tool Message =================================
Name: search_user_db

Maria tem 30 anos
================================== Ai Message ==================================
Tool Calls:
  sum_numbers (fc_bfb52655-5604-43bf-b75d-6dac3bc9ea19)
 Call ID: fc_bfb52655-5604-43bf-b75d-6dac3bc9ea19
  Args:
    numbers: [5, 10, 15]
================================= Tool Message =================================
Name: sum_numbers

A soma dos números [5, 10, 15] é 30
================================== Ai Message ==================================

A soma

### Agente Analista

In [20]:
import pandas as pd
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_groq import ChatGroq

In [21]:
df_temp = pd.read_csv("https://datahub.io/core/global-temp/r/0.csv")
df_temp.head()

,Source,Year,Mean
0,GCAG,1850,-0.4265
1,GCAG,1851,-0.2635
2,GCAG,1852,-0.2241
3,GCAG,1853,-0.2623
4,GCAG,1854,-0.2936


In [29]:
chat = ChatGroq(model="qwen/qwen3.8-27b")

agent_temp = create_pandas_dataframe_agent(
    chat,
    df_temp,
    agent_type="tool-calling",
    verbose=True,
    allow_dangerous_code=True
)

In [ ]:
# 1- Qual temperatura media global no ultimo ano

resposta = agent_temp.invoke({"input": "Qual foi a temperatura global média do ultimo ano?"})
print(resposta)



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': 'df.tail()'}`


      Source  Year    Mean
318  GISTEMP  2024  1.2817
319     GCAG  2024  1.1690
320  GISTEMP  2025  1.1850
321     GCAG  2025  1.0480
322     GCAG  2026  1.0421
Invoking: `python_repl_ast` with `{'query': "df[df['Year'] == df['Year'].max()]"}`


    Source  Year    Mean
322   GCAG  2026  1.0421

RateLimitError: Error code: 429 - {'error': {'message': "Request too large for model `qwen/qwen3.8-27b` in organization `org_01m4ba4j62eamr41w11nx3y4dz` service tier `on_demand` on output tokens per minute (OTPM): Limit 1000, Requested 1426. The request's expected output tokens exceed the enforced limit; reduce max_tokens (or the request's expected output) and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing", 'type': 'tokens', 'code': 'rate_limit_exceeded'}}